# 🎙️ Live Radio Stream Transcription with Whisper

## Overview
This notebook demonstrates how to:
1. Capture audio from live radio streams
2. Process audio into chunks
3. Transcribe using OpenAI's Whisper
4. Handle timing and context

## Architecture Decision Tree
```
Stream Type?
├─ HLS (m3u8) → ffmpeg capture
├─ HTTP Direct → requests streaming
└─ Icecast/Shoutcast → ffmpeg/streamlink

Processing Mode?
├─ Real-time (< 2s) → Complex buffering
├─ Near real-time (5-30s) → Chunk-based [We use this]
└─ Batch → Download then process

Model Size?
├─ CPU only → tiny/base
├─ GPU (T4) → small/medium [Recommended]
└─ High-end GPU → large
```

## 📦 Step 1: Install Dependencies

**What we're installing:**
- `openai-whisper`: Speech recognition model
- `ffmpeg-python`: Audio processing wrapper
- `pydub`: Audio manipulation library

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q openai-whisper ffmpeg-python pydub requests

## 🔍 Step 2: Find a Radio Stream URL

### How to Find Stream URLs:

**Method 1: Browser Dev Tools**
1. Go to radio station website
2. Open Developer Tools (F12)
3. Go to Network tab
4. Filter by "media" or "m3u8"
5. Play the stream
6. Copy the stream URL

**Method 2: Common Patterns**
- Look for URLs ending in `.m3u8`, `.pls`, `/stream`, `.mp3`
- Check station's "listen live" page source

**Example URLs (public streams):**

In [ ]:
# Example public radio stream URLs for testing
EXAMPLE_STREAMS = {
    'soma_fm_groove': 'http://ice1.somafm.com/groovesalad-128-mp3',
    'radio_paradise': 'http://stream.radioparadise.com/mp3-128',
    'wnyc': 'https://fm939.wnyc.org/wnycfm-web',
}

# YOU CAN REPLACE THIS WITH YOUR TARGET STREAM
STREAM_URL = EXAMPLE_STREAMS['soma_fm_groove']  # Change this to your stream

print(f"Target stream: {STREAM_URL}")
print("\n⚠️  To use a different stream:")
print("   1. Find the stream URL using browser dev tools")
print("   2. Replace STREAM_URL variable above")

## 🎯 Step 3: Capture Audio from Stream

### Capture Strategy Heuristics:

**Duration Selection:**
- **Test/Demo**: 30-120 seconds (fast, cheap)
- **Short program**: 5-15 minutes
- **Full show**: 30-60 minutes
- **All day**: Handle in chunks of 15-30 min

**Format Decisions:**
- WAV: Uncompressed, compatible, ~10MB/min
- MP3: Compressed, smaller, needs decoding
- **Recommendation**: WAV for processing, MP3 for storage

In [ ]:
import subprocess
import os
from datetime import datetime

def capture_stream(stream_url, duration_seconds=120, output_path='captured_audio.wav'):
    """
    Capture audio from a live stream using ffmpeg.
    
    Args:
        stream_url: URL of the radio stream
        duration_seconds: How long to capture (default: 120s = 2 min)
        output_path: Where to save the audio
    
    Heuristics:
        - 16kHz sample rate (Whisper native rate)
        - Mono audio (reduces size, sufficient for speech)
        - WAV format (uncompressed, fast to process)
    """
    
    print(f"📡 Capturing {duration_seconds}s from stream...")
    print(f"   URL: {stream_url}")
    print(f"   Output: {output_path}")
    
    cmd = [
        'ffmpeg',
        '-i', stream_url,           # Input stream
        '-t', str(duration_seconds), # Duration
        '-ar', '16000',              # Sample rate (Whisper uses 16kHz)
        '-ac', '1',                  # Mono audio
        '-y',                        # Overwrite output file
        output_path
    ]
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=duration_seconds + 30  # Add buffer for processing
        )
        
        if result.returncode == 0:
            file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
            print(f"✅ Capture complete! File size: {file_size_mb:.2f} MB")
            return output_path
        else:
            print(f"❌ Error capturing stream:")
            print(result.stderr)
            return None
            
    except subprocess.TimeoutExpired:
        print("⏱️  Capture timed out")
        return None
    except Exception as e:
        print(f"❌ Exception: {e}")
        return None

# Capture audio (adjust duration as needed)
CAPTURE_DURATION = 120  # 2 minutes for demo - increase for longer captures
audio_file = capture_stream(STREAM_URL, duration_seconds=CAPTURE_DURATION)

## ✂️ Step 4: Chunk Audio for Processing

### Chunking Strategy - Key Heuristics:

**Why Chunk?**
- Whisper processes 30s windows natively
- Long audio = high memory usage
- Enables parallel processing
- Better error handling

**Chunk Size Decision Matrix:**
```
Context Needed  │ Chunk Size │ Overlap
────────────────┼────────────┼─────────
News/Talk       │ 20-30s     │ 5s
Music           │ 30-60s     │ 10s  
Mixed           │ 30s        │ 5-7s
```

**Overlap Purpose:**
- Prevents word clipping at boundaries
- Provides context for better accuracy
- Trade-off: More processing, better results

In [ ]:
from pydub import AudioSegment
import math

def chunk_audio(audio_path, chunk_length_ms=30000, overlap_ms=5000):
    """
    Split audio into overlapping chunks for transcription.
    
    Args:
        audio_path: Path to audio file
        chunk_length_ms: Length of each chunk in milliseconds (default 30s)
        overlap_ms: Overlap between chunks in milliseconds (default 5s)
    
    Returns:
        List of (chunk_audio, start_time, end_time) tuples
    
    Heuristics:
        - 30s chunks match Whisper's native processing window
        - 5s overlap prevents boundary word clipping
        - Overlap = ~17% of chunk (good balance)
    """
    
    print(f"✂️  Chunking audio...")
    print(f"   Chunk size: {chunk_length_ms/1000}s")
    print(f"   Overlap: {overlap_ms/1000}s")
    
    # Load audio
    audio = AudioSegment.from_wav(audio_path)
    total_duration_ms = len(audio)
    
    chunks = []
    stride = chunk_length_ms - overlap_ms  # How far to advance each chunk
    
    for start_ms in range(0, total_duration_ms, stride):
        end_ms = min(start_ms + chunk_length_ms, total_duration_ms)
        chunk = audio[start_ms:end_ms]
        
        chunks.append({
            'audio': chunk,
            'start_time': start_ms / 1000,  # Convert to seconds
            'end_time': end_ms / 1000,
            'chunk_id': len(chunks)
        })
        
        if end_ms >= total_duration_ms:
            break
    
    print(f"   Created {len(chunks)} chunks")
    print(f"   Total duration: {total_duration_ms/1000:.1f}s")
    
    return chunks

# Create chunks
if audio_file:
    chunks = chunk_audio(audio_file, chunk_length_ms=30000, overlap_ms=5000)
else:
    print("⚠️  No audio file to chunk. Please capture audio first.")

## 🤖 Step 5: Load Whisper Model

### Model Selection Framework:

**Performance vs Quality Trade-offs:**

| Model  | Params | Speed (CPU) | Speed (GPU) | Quality | Use Case |
|--------|--------|-------------|-------------|---------|----------|
| tiny   | 39M    | Fast        | Very Fast   | 📊 70%  | Real-time, testing |
| base   | 74M    | Medium      | Fast        | 📊 75%  | CPU-only, quick |
| small  | 244M   | Slow        | Medium      | 📊 85%  | **Recommended** |
| medium | 769M   | Very Slow   | Slow        | 📊 92%  | High quality |
| large  | 1550M  | Extremely Slow | Medium   | 📊 95%  | Best quality |

**Decision Heuristics:**
- **Have GPU + want quality** → small or medium
- **CPU only** → tiny or base
- **Testing/prototyping** → base
- **Production** → small (best balance)
- **Non-English** → Use larger models

In [ ]:
import whisper
import torch

def load_whisper_model(model_size='base'):
    """
    Load Whisper model with automatic device selection.
    
    Args:
        model_size: 'tiny', 'base', 'small', 'medium', 'large'
    
    Heuristics:
        - Automatically uses GPU if available
        - Falls back to CPU gracefully
        - Base model good for testing, small for production
    """
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"🤖 Loading Whisper '{model_size}' model...")
    print(f"   Device: {device.upper()}")
    
    if device == 'cpu':
        print("   ⚠️  Using CPU. Consider enabling GPU in Colab: Runtime > Change runtime type")
    
    model = whisper.load_model(model_size, device=device)
    print(f"   ✅ Model loaded successfully")
    
    return model

# Load model - change to 'small' or 'medium' for better quality (requires GPU)
MODEL_SIZE = 'base'  # Change to 'small' if you have GPU enabled
whisper_model = load_whisper_model(MODEL_SIZE)

print("\n💡 Model Size Guide:")
print("   base   → Fast, good for testing (recommended for CPU)")
print("   small  → Better quality, needs GPU (recommended for production)")
print("   medium → High quality, slower (needs good GPU)")

## 🎯 Step 6: Transcribe Audio

### Transcription Strategy:

**Processing Pipeline:**
```
Chunk → Temp File → Whisper → Text + Timestamps → Deduplicate → Output
```

**Key Optimizations:**
1. **Language hint**: Speeds up by 20-40% if you know the language
2. **FP16**: Faster on GPU, minimal quality loss
3. **Batch processing**: Process multiple chunks in parallel (not shown)

**Deduplication Strategy:**
- Overlapping chunks produce duplicate text
- Keep text from first occurrence
- Simple but effective for continuous speech

In [ ]:
import tempfile
import os
from collections import defaultdict

def transcribe_chunks(chunks, model, language=None):
    """
    Transcribe audio chunks with timing information.
    
    Args:
        chunks: List of audio chunks from chunk_audio()
        model: Loaded Whisper model
        language: Optional language code (e.g., 'en', 'es', 'fr')
                  Specifying language speeds up transcription by 20-40%
    
    Returns:
        List of transcription results with timestamps
    
    Heuristics:
        - Process chunks sequentially (simple, reliable)
        - Use temporary files (avoids memory issues)
        - Include timing info for sync with original audio
        - Language hint = significant speedup if known
    """
    
    results = []
    
    print(f"\n🎯 Transcribing {len(chunks)} chunks...")
    if language:
        print(f"   Language: {language} (this speeds up processing)")
    
    for i, chunk_data in enumerate(chunks):
        print(f"   [{i+1}/{len(chunks)}] Processing chunk at {chunk_data['start_time']:.1f}s...", end=' ')
        
        # Export chunk to temporary file
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            chunk_data['audio'].export(tmp.name, format='wav')
            tmp_path = tmp.name
        
        try:
            # Transcribe with Whisper
            result = model.transcribe(
                tmp_path,
                language=language,
                fp16=torch.cuda.is_available(),  # Use FP16 on GPU for speed
            )
            
            results.append({
                'chunk_id': chunk_data['chunk_id'],
                'start_time': chunk_data['start_time'],
                'end_time': chunk_data['end_time'],
                'text': result['text'].strip(),
                'segments': result.get('segments', []),
                'language': result.get('language', 'unknown')
            })
            
            print(f"✓ ({result.get('language', 'unknown')})")
            
        finally:
            # Clean up temp file
            if os.path.exists(tmp_path):
                os.unlink(tmp_path)
    
    print(f"\n✅ Transcription complete!")
    return results

def deduplicate_transcripts(results):
    """
    Remove duplicate text from overlapping chunks.
    
    Simple strategy: Keep text from first occurrence.
    More sophisticated: Use word-level deduplication (not implemented).
    """
    seen_text = set()
    deduplicated = []
    
    for result in results:
        # Simple sentence-level deduplication
        sentences = result['text'].split('. ')
        unique_sentences = []
        
        for sentence in sentences:
            sentence = sentence.strip()
            if sentence and sentence not in seen_text:
                seen_text.add(sentence)
                unique_sentences.append(sentence)
        
        if unique_sentences:
            deduplicated.append({
                **result,
                'text': '. '.join(unique_sentences)
            })
    
    return deduplicated

# Transcribe
if audio_file and chunks:
    # Set language if you know it (speeds up by 20-40%)
    LANGUAGE = 'en'  # Change to None for auto-detect, or 'es', 'fr', etc.
    
    transcriptions = transcribe_chunks(chunks, whisper_model, language=LANGUAGE)
    transcriptions = deduplicate_transcripts(transcriptions)
else:
    print("⚠️  No chunks available. Please complete previous steps.")

## 📊 Step 7: Display Results

### Output Format Considerations:

**Format Options:**
1. **Timestamped Text** (we use this) → Good for review
2. **SRT/VTT** → Subtitle format
3. **JSON** → Machine-readable
4. **Plain Text** → Simple concatenation

**Use Cases:**
- Monitoring → Timestamped text + keyword search
- Archival → JSON with full metadata
- Subtitles → SRT format
- Analysis → JSON for further processing

In [ ]:
def format_timestamp(seconds):
    """Convert seconds to HH:MM:SS format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def display_transcription(transcriptions):
    """
    Display transcription with timestamps in a readable format.
    """
    print("\n" + "="*70)
    print("📝 TRANSCRIPTION RESULTS")
    print("="*70 + "\n")
    
    for result in transcriptions:
        start = format_timestamp(result['start_time'])
        end = format_timestamp(result['end_time'])
        
        print(f"[{start} - {end}]")
        print(f"{result['text']}")
        print()
    
    print("="*70)

def save_transcription(transcriptions, output_path='transcription.txt'):
    """
    Save transcription to a file.
    """
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("LIVE RADIO TRANSCRIPTION\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*70 + "\n\n")
        
        for result in transcriptions:
            start = format_timestamp(result['start_time'])
            end = format_timestamp(result['end_time'])
            f.write(f"[{start} - {end}]\n")
            f.write(f"{result['text']}\n\n")
    
    print(f"💾 Transcription saved to: {output_path}")
    return output_path

# Display and save results
if 'transcriptions' in locals() and transcriptions:
    display_transcription(transcriptions)
    save_transcription(transcriptions)
    
    # Statistics
    total_words = sum(len(r['text'].split()) for r in transcriptions)
    total_time = transcriptions[-1]['end_time'] if transcriptions else 0
    
    print(f"\n📊 Statistics:")
    print(f"   Total chunks: {len(transcriptions)}")
    print(f"   Total words: {total_words}")
    print(f"   Duration: {format_timestamp(total_time)}")
    if total_time > 0:
        print(f"   Words/minute: {int(total_words / (total_time / 60))}")
else:
    print("⚠️  No transcriptions available.")

## 🚀 Step 8: Advanced - Real-Time Processing (Optional)

### Continuous Processing Architecture:

```python
# Pseudo-code for production system
while True:
    # 1. Capture chunk (background thread)
    audio_chunk = capture_stream_chunk(duration=10)  # 10s chunks
    
    # 2. Add to buffer with overlap
    buffer.add(audio_chunk, overlap=5)  # 5s overlap
    
    # 3. Process when buffer full (30s)
    if buffer.is_ready():
        result = transcribe(buffer.get())
        
        # 4. Store/process result
        store_to_database(result)
        check_keywords(result)  # Alert on keywords
        
        # 5. Shift buffer
        buffer.slide(25)  # Keep last 5s for next chunk
```

### Key Challenges:
1. **Latency**: 10-30s typical (capture + process + buffer)
2. **Memory**: Need to manage buffer sizes
3. **Stability**: Handle stream interruptions
4. **Sync**: Keep transcription aligned with audio

### Production Recommendations:
- Use `faster-whisper` (2-4x speedup)
- Implement proper logging
- Add keyword detection/alerting
- Store in database (SQLite/PostgreSQL)
- Monitor with Prometheus/Grafana
- Use message queue (RabbitMQ/Redis) for scaling

In [ ]:
# Example: Keyword Detection Function
def detect_keywords(transcription_text, keywords):
    """
    Simple keyword detection in transcription.
    
    For production:
    - Use fuzzy matching (fuzzywuzzy)
    - Implement context extraction
    - Add phonetic matching
    - Send alerts (email, SMS, webhook)
    """
    text_lower = transcription_text.lower()
    found_keywords = [kw for kw in keywords if kw.lower() in text_lower]
    return found_keywords

# Example usage
KEYWORDS_TO_MONITOR = ['weather', 'traffic', 'news', 'breaking']  # Customize

if 'transcriptions' in locals() and transcriptions:
    print("\n🔍 Keyword Detection Example:\n")
    for result in transcriptions:
        found = detect_keywords(result['text'], KEYWORDS_TO_MONITOR)
        if found:
            print(f"⚠️  Found keywords {found} at {format_timestamp(result['start_time'])}")
            print(f"   Text: {result['text'][:100]}...\n")

## 📚 Next Steps & Resources

### Improvement Path:

**Level 1 - Current (MVP)** ✅
- Capture sample audio
- Chunk and transcribe
- Basic output

**Level 2 - Enhanced**
- Use `faster-whisper` for 2-4x speedup
- Add speaker diarization
- Implement keyword alerting
- Better deduplication

**Level 3 - Production**
- Continuous capture (24/7)
- Database storage
- Web interface
- API endpoints
- Monitoring/logging

### Key Libraries for Production:
- `faster-whisper`: 2-4x faster than openai-whisper
- `whisper-streaming`: Real-time capable
- `pyannote.audio`: Speaker diarization
- `streamlink`: Better stream handling
- `celery`: Task queue for scaling

### Helpful Resources:
- [Whisper GitHub](https://github.com/openai/whisper)
- [faster-whisper](https://github.com/guillaumekln/faster-whisper)
- [whisper-streaming](https://github.com/ufal/whisper_streaming)
- [FFmpeg Documentation](https://ffmpeg.org/documentation.html)

## 🎯 Quick Reference: Common Commands

```python
# Find stream URL using browser:
# 1. Open station website
# 2. F12 → Network tab
# 3. Filter by 'media' or 'm3u8'
# 4. Play stream
# 5. Copy URL from network request

# Test stream URL:
!ffplay -i "YOUR_STREAM_URL" -t 10  # Play 10 seconds

# Capture longer duration:
capture_stream(STREAM_URL, duration_seconds=3600)  # 1 hour

# Use better model (needs GPU):
whisper_model = load_whisper_model('small')  # or 'medium'

# Specify language (20-40% speedup):
transcriptions = transcribe_chunks(chunks, whisper_model, language='en')

# Check GPU availability:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
```